In [5]:
import os
import copy
import json
import pickle as pkl
import pandas as pd
from tqdm import tqdm
import random as random
import numpy as np
from ray import tune
import numpy as np
from sklearn import metrics

In [6]:
    neg_to_pos_ratio = 1

    results_file = "results_titles_abstracts_gbdt.json"

    inputs = "titles_abstracts"

    filename_pos_human = "../data/pubmed_articles_pos_human.csv"
    filename_neg_human = "../data/pubmed_articles_neg_human.csv"

    with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
        prop_typ_id_dict = pkl.load(f)
    ptype = 'is_sr'

In [28]:
pos_test_ids = []
for ptype in prop_typ_id_dict.keys():
    pos_test_ids.extend(prop_typ_id_dict[ptype]['pos']['test'])

In [31]:
neg_test_ids = []
for ptype in prop_typ_id_dict.keys():
    neg_test_ids.extend(prop_typ_id_dict[ptype]['neg']['test'])

In [8]:
pos_data_human = pd.read_csv(filename_pos_human, chunksize=100000)
neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)

In [32]:
pos_test_set = pd.DataFrame()
for pos_chunk in tqdm(pos_data_human):

                indices = pos_chunk.index
                indices = list(indices)

                pos_test_indices = list(set(indices).intersection(set(pos_test_ids)))

                pos_test_subset = pos_chunk.loc[pos_test_indices]
                if pos_test_set.empty:
                    pos_test_set = pd.DataFrame(columns=pos_test_set.columns)
                pos_test_set = pd.concat([pos_test_set, pos_test_subset])

18it [00:24,  1.36s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
50it [01:07,  1.34s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
85it [01:53,  1.33s/it]


In [44]:
neg_test_set = pd.DataFrame()
for neg_chunk in tqdm(neg_data_human):

                indices = neg_chunk.index
                indices = list(indices)

                neg_test_indices = list(set(indices).intersection(set(neg_test_ids)))

                neg_test_subset = neg_chunk.loc[neg_test_indices]
                if neg_test_set.empty:
                    neg_test_set = pd.DataFrame(columns=neg_test_set.columns)
                neg_test_set = pd.concat([neg_test_set, neg_test_subset])

56it [01:24,  1.52s/it]


In [40]:
pos_test_set_random_sample_1000 = pos_test_set.sample(1000)

In [43]:
pos_test_set_random_sample_1000.to_pickle("pos_test_set_random_sample_1000.pkl")

In [45]:
neg_test_set_random_sample_1000 = neg_test_set.sample(1000)

In [46]:
neg_test_set_random_sample_1000.to_pickle("neg_test_set_random_sample_1000.pkl")

In [47]:
pos_test_set_random_sample_1000

,Unnamed: 0,unnamed: 0,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
1573959,1573959,22468,16777475,MEDLINE,Human,Micronucleus frequency in women with genital C...,"[{'header': '_UNSTRUCTURED', 'text': 'The main...",The main aim of the present study was to inves...,"[{'Initials': 'A', 'LastName': 'Dimitrijević',...",Mutation research,Mutat Res,2006.0,"['Adult', 'Anti-Bacterial Agents', 'Azithromyc...",Sep,608,1,"{'page_from': '43', 'page_to': '48'}","['Clinical Trial', 'Journal Article', ""Researc...",[],[]
2932056,2932056,20161,10655012,MEDLINE,Human,Length and number of hospitalizations in two c...,"[{'header': '_UNSTRUCTURED', 'text': 'Two coho...",Two cohorts of male veteran patients with schi...,"[{'Initials': 'G', 'LastName': 'Goldstein', 'F...","Psychiatric services (Washington, D.C.)",Psychiatr Serv,2000.0,"['Adult', 'Cohort Studies', 'Hospitalization',...",Feb,51,2,"{'page_from': '245', 'page_to': '247'}","['Comparative Study', 'Journal Article', ""Rese...",[],[]
2383111,2383111,6271,10978816,MEDLINE,Human,Part 12: from science to survival. Strengtheni...,"[{'header': '_UNSTRUCTURED', 'text': 'Cardiac ...",Cardiac arrest treatment continues to evolve. ...,[],Resuscitation,Resuscitation,2000.0,"['Algorithms', 'Cardiopulmonary Resuscitation'...",Aug,46,1-3,"{'page_from': '417', 'page_to': '430'}","['Consensus Development Conference', 'Guidelin...",[],[]
5056979,5056979,3313,8403900,MEDLINE,Human,Another look at the Dalkon Shield: meta-analys...,"[{'header': '_UNSTRUCTURED', 'text': ""Numerous...",Numerous non-comparative clinical trials of th...,"[{'Initials': 'I', 'LastName': 'Sivin', 'ForeN...",Contraception,Contraception,1993.0,"['Case-Control Studies', 'Female', 'Humans', '...",Jul,48,1,"{'page_from': '1', 'page_to': '12'}","['Clinical Trial', 'Comparative Study', 'Journ...",[],[]
2407023,2407023,26611,31292671,MEDLINE,Human,Incomplete tumor capsule on preoperative imagi...,"[{'header': 'PURPOSE', 'text': 'Microvascular ...","PURPOSE\n\n\nMicrovascular invasion (MVI), whi...","[{'Initials': 'F', 'LastName': 'Zhu', 'ForeNam...",Abdominal radiology (New York),Abdom Radiol (NY),2019.0,"['Carcinoma, Hepatocellular', 'Humans', 'Liver...",09,44,9,"{'page_from': '3049', 'page_to': '3057'}","['Journal Article', 'Meta-Analysis', 'Systemat...",[],['10.1007/s00261-019-02126-9']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2168920,2168920,26507,16109111,MEDLINE,Human,Sumatriptan and naproxen sodium for the acute ...,"[{'header': 'OBJECTIVE', 'text': 'To evaluate ...",OBJECTIVE\n\n\nTo evaluate the efficacy and to...,"[{'Initials': 'TR', 'LastName': 'Smith', 'Fore...",Headache,Headache,2005.0,"['Acute Disease', 'Adult', 'Anti-Inflammatory ...",Sep,45,8,"{'page_from': '983', 'page_to': '991'}","['Journal Article', 'Multicenter Study', 'Rand...",[],[]
7134439,7134439,24488,21389011,MEDLINE,Human,Water disinfection by-products and bladder can...,"[{'header': '_UNSTRUCTURED', 'text': 'Several ...",Several epidemiological studies suggested an a...,"[{'Initials': 'N', 'LastName': 'Costet', 'Fore...",Occupational and environmental medicine,Occup Environ Med,2011.0,"['Adult', 'Aged', 'Aged, 80 and over', 'Case-C...",May,68,5,"{'page_from': '379', 'page_to': '385'}","['Journal Article', 'Meta-Analysis', ""Research...",[],['10.1136/oem.2010.062703']
557621,557621,11785,12765053,MEDLINE,Human,[Prostate cancer screening].,"[{'header': '_UNSTRUCTURED', 'text': ""Prostate...",Prostate cancer has become the most frequent c...,"[{'Initials': 'A', 'LastName': 'Villers', 'For...",Progres en urologie : journal de l'Association...,Prog Urol,2003.0,"['Humans', 'Male', 'Mass Screening', 'Prostate...",Apr,13,2,"{'page_from': '209', 'page_to': '214'}","['English Abstract', 'Guideline', 'Journal Art...",[],[]
6799560,6799560,563,19280874,MEDLINE,Human,[Guidelines for the general management of pati...,"[{'header':

In [48]:
neg_test_set_random_sample_1000

,Unnamed: 0.1,index,Unnamed: 0,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,...,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
4960936,4960936,5557194,29960,21298449,MEDLINE,Human,Coping processes relevant to posttraumatic gro...,"[{'header': 'OBJECTIVE', 'text': 'This study a...",OBJECTIVE\n\n\nThis study aims to perform an e...,"[{'Initials': 'RK', 'LastName': 'Rajandram', '...",...,Support Care Cancer,2011.0,"['Adaptation, Psychological', 'Evidence-Based ...",May,19,5,"{'page_from': '583', 'page_to': '589'}","['Journal Article', 'Review']",[],['10.1007/s00520-011-1105-0']
2618533,2618533,2934235,5815,25212694,MEDLINE,Human,The dual peroxisome proliferator-activated rec...,"[{'header': '_UNSTRUCTURED', 'text': 'We repor...",We report here the efficacy and safety of GFT5...,"[{'Initials': 'R', 'LastName': 'Hanf', 'ForeNa...",...,Diab Vasc Dis Res,2014.0,"['Animals', 'Blood Glucose', 'Chalcones', 'Dia...",Nov,11,6,"{'page_from': '440', 'page_to': '447'}","['Journal Article', ""Research Support, Non-U.S...",[],['10.1177/1479164114548027']
3920728,3920728,4388633,5768,27259657,MEDLINE,Human,The Orthology Ontology: development and applic...,"[{'header': 'BACKGROUND', 'text': 'Computation...",BACKGROUND\n\n\nComputational comparative anal...,"[{'Initials': 'JT', 'LastName': 'Fernández-Bre...",...,J Biomed Semantics,2016.0,"['Gene Ontology', 'Genomics', 'Reference Stand...",06,7,1,"{'page_from': '34', 'page_to': '34'}","['Journal Article', ""Research Support, Non-U.S...",[],['10.1186/s13326-016-0077-x']
3029326,3029326,3392512,29477,19707510,PubMed-not-MEDLINE,Human,Multimodal treatment of primary extraskeletal ...,"[{'header': '_UNSTRUCTURED', 'text': ""Extraske...",Extraskeletal Ewing's sarcoma (EES) is a type ...,"[{'Initials': 'WS', 'LastName': 'Lee', 'ForeNa...",...,Cancer Res Treat,2009.0,[],Jun,41,2,"{'page_from': '108', 'page_to': '112'}",['Journal Article'],[],['10.4143/crt.2009.41.2.108']
4608795,4608795,5165756,25130,32090377,PubMed-not-MEDLINE,Human,Structural Elucidation of Silver(I) Amides and...,"[{'header': '_UNSTRUCTURED', 'text': 'This stu...",This study details the isolation and character...,"[{'Initials': 'SA', 'LastName': 'Orr', 'ForeNa...",...,Chemistry,2020.0,[],Apr,26,22,"{'page_from': '4947', 'page_to': '4951'}",['Journal Article'],[],['10.1002/chem.202000169']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4577543,4577543,5125443,23423,28240097,MEDLINE,Human,Soluble intercellular adhesion molecule-1: a p...,"[{'header': 'AIM', 'text': ""Pain therapy is st...",AIM\n\n\nPain therapy is strongly guided by pa...,"[{'Initials': 'B', 'LastName': 'Luchting', 'Fo...",...,Biomark Med,2017.0,"['Adult', 'Aged', 'Aged, 80 and over', 'Area U...",Mar,11,3,"{'page_from': '265', 'page_to': '276'}",['Journal Article'],[],['10.2217/bmm-2016-0246']
1907152,1907152,2117047,12908,8293488,MEDLINE,Human,HRT and osteoporosis.,"[{'header': '_UNSTRUCTURED', 'text': 'Hormone ...",Hormone replacement therapy is well documented...,"[{'Initials': 'DH', 'LastName': 'Barlow', 'For...",...,Baillieres Clin Rheumatol,1993.0,"['Aging', 'Bone and Bones', 'Cardiovascular Di...",Oct,7,3,"{'page_from': '535', 'page_to': '548'}","['Journal Article', 'Review']",[],[]
3661739,3661739,4095911,26426,17807610,PubMed-not-MEDLINE,Human,Geomagnetic origin for transient particle even...,"[{'header': '_UNSTRUCTURED', 'text': 'Transien...",Transient events observed since 1980 by the Ga...,"[{'Initials': 'GH', 'LastName': 'Share', 'Fore...",...,Science,1989.0,[],Apr,244,4903,"{'page_from': '444', 'page_to': '448'}",['Journal Article'],[],[]
3580486,3580486,4006353,2725,28992387,MEDLINE,Human,Nationwide Experience With Off-Label Use of In...,"[{'header': 'OBJECTIVE', 'text': 'Approximatel...",OBJECTIVE\n\n\nApproximately 30-45% of patient...,"[{'Initials': 'S', 'LastName': 'Akar', 'ForeNa...",...,Arthritis Care Res (Hoboken),2018.0,"['Adolescent', 'Adult', 'Aged', 'Drug

In [49]:
!pwd

/home/k2141255/research/study_design_classifier/study-design-classifier
